# Real-Time Drowsiness Test (Webcam)

This notebook runs live prediction from webcam using the same feature idea as your training pipeline:
- EAR
- MAR
- pitch
- yaw

Press `q` to quit the live window.

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from collections import deque, Counter

try:
    import mediapipe as mp
except ImportError:
    raise ImportError("mediapipe not installed. Run: pip install mediapipe")

try:
    import joblib
except ImportError:
    raise ImportError("joblib not installed. Run: pip install joblib")

MODEL_PATH = "random_forest_model.pkl"
LE_PATH = "label_encoder.pkl"
COLS_PATH = "feature_columns.pkl"

for p in [MODEL_PATH, LE_PATH, COLS_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Required file not found: {p}")

model = joblib.load(MODEL_PATH)
le = joblib.load(LE_PATH)
feature_columns = joblib.load(COLS_PATH)

print("Model loaded")
print("Classes:", list(le.classes_))
print("Feature columns:", feature_columns)

c:\Users\PMLS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Model loaded
Classes: ['Alert', 'Drowsy', 'Low Vigilance']
Feature columns: ['EAR_mean', 'EAR_std', 'EAR_min', 'MAR_mean', 'MAR_max', 'pitch_mean', 'yaw_mean', 'EAR_dev_mean', 'MAR_dev_mean']


c:\Users\PMLS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.6.1 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\PMLS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:
# Parameters (tune here)
WEBCAM_INDEX = 0
WINDOW_SIZE = 15
SMOOTHING_WINDOW = 12
SAMPLE_EVERY_N_FRAMES = 3
CALIBRATION_VALID_FRAMES = 60

# Landmark indices
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
MOUTH_TOP, MOUTH_BOTTOM, MOUTH_LEFT, MOUTH_RIGHT = 13, 14, 78, 308
NOSE, CHIN, LEFT_FACE, RIGHT_FACE = 1, 152, 234, 454

In [4]:
mp_face_mesh = mp.solutions.face_mesh

def compute_EAR(landmarks_xy, eye):
    p1, p2, p3, p4, p5, p6 = [np.array(landmarks_xy[i][:2]) for i in eye]
    vertical1 = np.linalg.norm(p2 - p6)
    vertical2 = np.linalg.norm(p3 - p5)
    horizontal = np.linalg.norm(p1 - p4)
    return (vertical1 + vertical2) / (2.0 * horizontal + 1e-6)

def compute_MAR(landmarks_xy):
    top = np.array(landmarks_xy[MOUTH_TOP][:2])
    bottom = np.array(landmarks_xy[MOUTH_BOTTOM][:2])
    left = np.array(landmarks_xy[MOUTH_LEFT][:2])
    right = np.array(landmarks_xy[MOUTH_RIGHT][:2])
    vertical = np.linalg.norm(top - bottom)
    horizontal = np.linalg.norm(left - right)
    return vertical / (horizontal + 1e-6)

def compute_head_pose_approx(landmarks_xy):
    nose = np.array(landmarks_xy[NOSE][:2])
    chin = np.array(landmarks_xy[CHIN][:2])
    left = np.array(landmarks_xy[LEFT_FACE][:2])
    right = np.array(landmarks_xy[RIGHT_FACE][:2])
    pitch = np.linalg.norm(nose - chin)
    yaw = np.linalg.norm(left - right)
    return pitch, yaw

def make_window_row(buffer, baseline_ear, baseline_mar):
    df = pd.DataFrame(list(buffer))
    df["EAR_dev"] = df["EAR"] - baseline_ear
    df["MAR_dev"] = df["MAR"] - baseline_mar
    row = {
        "EAR_mean": df["EAR"].mean(),
        "EAR_std": df["EAR"].std(),
        "EAR_min": df["EAR"].min(),
        "MAR_mean": df["MAR"].mean(),
        "MAR_max": df["MAR"].max(),
        "pitch_mean": df["pitch"].mean(),
        "yaw_mean": df["yaw"].mean(),
        "EAR_dev_mean": df["EAR_dev"].mean(),
        "MAR_dev_mean": df["MAR_dev"].mean()
    }
    X = pd.DataFrame([row])
    X = X[feature_columns]
    return X


In [5]:
def run_realtime():
    cap = cv2.VideoCapture(WEBCAM_INDEX)
    if not cap.isOpened():
        print("Could not open webcam")
        return

    print("Webcam opened")
    print("Calibration: keep neutral/alert face, look at camera")

    frame_count = 0
    calibration = []

    feature_buffer = deque(maxlen=WINDOW_SIZE)
    pred_buffer = deque(maxlen=SMOOTHING_WINDOW)

    current_pred = "Collecting..."
    current_conf = 0.0
    baseline_ear = None
    baseline_mar = None

    with mp_face_mesh.FaceMesh(
        static_image_mode=False,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as face_mesh:

        # Phase 1: calibration
        while len(calibration) < CALIBRATION_VALID_FRAMES:
            ret, frame = cap.read()
            if not ret:
                print("Failed to read webcam frame")
                cap.release()
                cv2.destroyAllWindows()
                return

            frame = cv2.flip(frame, 1)
            frame_count += 1
            disp = frame.copy()

            if frame_count % SAMPLE_EVERY_N_FRAMES == 0:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                res = face_mesh.process(rgb)
                if res.multi_face_landmarks:
                    h, w, _ = frame.shape
                    fl = res.multi_face_landmarks[0]
                    lms = [(int(lm.x * w), int(lm.y * h), lm.z) for lm in fl.landmark]
                    try:
                        left_ear = compute_EAR(lms, LEFT_EYE)
                        right_ear = compute_EAR(lms, RIGHT_EYE)
                        ear = (left_ear + right_ear) / 2.0
                        mar = compute_MAR(lms)
                        calibration.append({"EAR": ear, "MAR": mar})
                    except Exception:
                        pass

            cv2.putText(disp, f"Calibration: {len(calibration)}/{CALIBRATION_VALID_FRAMES}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
            cv2.putText(disp, "Stay neutral and look at camera", (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            cv2.putText(disp, "Press q to quit", (20, disp.shape[0] - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            cv2.imshow("Realtime Drowsiness Test", disp)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                cap.release()
                cv2.destroyAllWindows()
                return

        cal_df = pd.DataFrame(calibration)
        baseline_ear = cal_df["EAR"].mean()
        baseline_mar = cal_df["MAR"].mean()
        print(f"Calibration done. baseline_EAR={baseline_ear:.4f}, baseline_MAR={baseline_mar:.4f}")

        # Phase 2: realtime prediction
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)
            frame_count += 1
            disp = frame.copy()

            if frame_count % SAMPLE_EVERY_N_FRAMES == 0:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                res = face_mesh.process(rgb)

                if res.multi_face_landmarks:
                    h, w, _ = frame.shape
                    fl = res.multi_face_landmarks[0]
                    lms = [(int(lm.x * w), int(lm.y * h), lm.z) for lm in fl.landmark]

                    # Draw landmark dots
                    for (x, y, _) in lms:
                        cv2.circle(disp, (x, y), 1, (0, 255, 0), -1)

                    try:
                        left_ear = compute_EAR(lms, LEFT_EYE)
                        right_ear = compute_EAR(lms, RIGHT_EYE)
                        ear = (left_ear + right_ear) / 2.0
                        mar = compute_MAR(lms)
                        pitch, yaw = compute_head_pose_approx(lms)

                        feature_buffer.append({
                            "EAR": ear,
                            "MAR": mar,
                            "pitch": pitch,
                            "yaw": yaw
                        })

                        if len(feature_buffer) == WINDOW_SIZE:
                            X_live = make_window_row(feature_buffer, baseline_ear, baseline_mar)
                            pred_id = model.predict(X_live)[0]
                            pred_label = le.inverse_transform([pred_id])[0]
                            pred_buffer.append(pred_label)
                            current_pred = Counter(pred_buffer).most_common(1)[0][0]

                            if hasattr(model, "predict_proba"):
                                probs = model.predict_proba(X_live)[0]
                                current_conf = float(np.max(probs) * 100.0)

                    except Exception:
                        pass

            if current_pred == "Alert":
                color = (0, 255, 0)
            elif current_pred == "Low Vigilance":
                color = (0, 255, 255)
            elif current_pred == "Drowsy":
                color = (0, 0, 255)
            else:
                color = (255, 255, 255)

            cv2.putText(disp, f"Prediction: {current_pred}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
            cv2.putText(disp, f"Confidence: {current_conf:.1f}%", (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2)
            cv2.putText(disp, f"Feature buffer: {len(feature_buffer)}/{WINDOW_SIZE}", (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(disp, "Press q to quit", (20, disp.shape[0] - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            cv2.imshow("Realtime Drowsiness Test", disp)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()
    print("Webcam closed")

In [7]:
run_realtime()

Webcam opened
Calibration: keep neutral/alert face, look at camera


c:\Users\PMLS\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Calibration done. baseline_EAR=0.3108, baseline_MAR=0.0037


c:\Users\PMLS\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\PMLS\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\PMLS\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.G

Webcam closed
